In [ ]:
# ==============================================================================
# MULTIVIEW LEAST SQUARES SUPPORT VECTOR MACHINE (MV-LSSVM) EXPERIMENT
# Subsample Budget 'm' = Ratio/Fraction of Training Dataset Size
#
# Strategies evaluated:
# 1. MV-LSSVM: All samples (100% baseline)
# 2. MV-LSSVM: Global Random (Same sample fraction removed across all views)
# 3. MV-LSSVM: Uniform Random (Per view)
# 4. MV-LSSVM: Nyström (Per view)
# 5. MV-LSSVM: Unsupervised clustering (m-fraction centroids per view)
# 6. MV-LSSVM: Stratified Kernel Herding (Per view)
# 7. MV-LSSVM: Proposed Stratified HSIC Herding + Warmup CV (Per view)
# ==============================================================================
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.cluster import KMeans
from sklearn.datasets import load_digits
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import OneHotEncoder


# --- 0. Output Directory Setup ---

OUTPUT_DIR = "9_mvlssvm_ratio_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"[INFO] Artifacts will be saved to: {os.path.abspath(OUTPUT_DIR)}")


# --- 1. MV-LSSVM Model Classifier ---


class Multiclass_MV_LSSVM(BaseEstimator, ClassifierMixin):
    """Multiclass Multi-View Least Squares Support Vector Machine (MV-LSSVM).

    This classifier trains one LSSVM model using multiple feature views.
    Each view may use a different RBF kernel parameter and a potentially
    different subset of training samples. The individual kernel matrices
    are combined into a block-diagonal kernel matrix during training.

    Attributes:
        classes_: Sorted unique class labels observed during training.
        sub_X_: Training samples selected for each view.
        sub_Y_: One-hot encoded training targets transformed to {-1, +1}.
        alpha_: LSSVM regression coefficients for the block kernel matrix.
    """

    def __init__(self, kernel_gammas, selected_indices, C=1.0):
        """Initializes the multiclass MV-LSSVM classifier.

        Args:
            kernel_gammas: RBF kernel gamma value for each feature view.
            selected_indices: Training sample indices selected for each view.
                Is a dictionary mapping view names to index arrays.
            C: LSSVM regularization parameter. Higher values apply weaker
                regularization.
        """
        self.kernel_gammas = kernel_gammas
        self.selected_indices = selected_indices
        self.C = C

    def fit(self, X_list, y):
        """Fits the MV-LSSVM model using the provided training views.

        Each view is reduced to the samples specified by `selected_indices`.
        The corresponding RBF kernel matrices are then assembled into a
        block-diagonal matrix. Multiclass targets are represented using
        {-1, +1} coding before solving the LSSVM linear system.

        Args:
            X_list: List of feature matrices, one matrix per view. Each matrix
                has shape (n_samples, n_features_view).
            y: One-dimensional array containing the class labels.

        Returns:
            Multiclass_MV_LSSVM: The fitted classifier.
        """
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)
        n_views = len(X_list)

        # Convert class labels to a one-hot representation and then
        # transform binary indicators from {0, 1} to {-1, +1}.
        ohe = OneHotEncoder(sparse_output=False)
        Y_onehot = ohe.fit_transform(y.reshape(-1, 1))
        Y_onehot = np.where(Y_onehot == 0, -1.0, 1.0)

        self.sub_X_ = []
        self.sub_Y_ = []

        K_blocks = []
        total_samples = 0

        # Build one kernel matrix for the selected samples of each view.
        for v in range(n_views):
            idx_v = np.array(self.selected_indices[v], dtype=int)
            X_v_sub = X_list[v][idx_v]
            Y_v_sub = Y_onehot[idx_v]

            self.sub_X_.append(X_v_sub)
            self.sub_Y_.append(Y_v_sub)

            n_v = len(idx_v)
            total_samples += n_v

            # Compute the RBF kernel using the gamma associated with
            # the current view.
            Kv = rbf_kernel(X_v_sub, gamma=self.kernel_gammas[v])
            K_blocks.append(Kv)

        # Allocate the combined block-diagonal kernel matrix and
        # corresponding target matrix.
        K_tilde = np.zeros((total_samples, total_samples))
        Y_tilde = np.zeros((total_samples, n_classes))

        current_idx = 0

        # Insert each view-specific kernel matrix into the corresponding
        # diagonal block of the combined kernel matrix.
        for v in range(n_views):
            n_v = len(self.selected_indices[v])
            sl = slice(current_idx, current_idx + n_v)

            K_tilde[sl, sl] = K_blocks[v]
            Y_tilde[sl, :] = self.sub_Y_[v]

            current_idx += n_v

        # Solve the regularized LSSVM linear system:
        #
        #     (K + I/C) alpha = Y
        #
        # The identity term provides the LSSVM regularization.
        A = K_tilde + (1.0 / self.C) * np.eye(total_samples)
        self.alpha_ = np.linalg.solve(A, Y_tilde)

        return self

    def predict(self, X_list):
        """Predicts class labels for samples from multiple feature views.

        For each view, an RBF kernel is computed between the test samples
        and the selected training samples. The resulting decision scores
        are accumulated across views, and the class with the largest
        aggregated score is returned.

        Args:
            X_list: List of feature matrices, one matrix per view. All
                matrices must contain the same number of samples.

        Returns:
            np.ndarray: Predicted class label for each input sample.
        """
        n_samples = X_list[0].shape[0]
        n_classes = len(self.classes_)
        n_views = len(X_list)

        # Accumulate multiclass decision scores across all views.
        f_scores = np.zeros((n_samples, n_classes))

        current_idx = 0

        for v in range(n_views):
            n_v = len(self.selected_indices[v])
            sl = slice(current_idx, current_idx + n_v)

            # Compute the test-to-selected-training kernel matrix
            # for the current view.
            K_test_v = rbf_kernel(
                X_list[v],
                self.sub_X_[v],
                gamma=self.kernel_gammas[v],
            )

            # Add the contribution of the current view to the
            # overall multiclass decision scores.
            f_scores += K_test_v @ self.alpha_[sl, :]
            current_idx += n_v

        # Select the class corresponding to the maximum decision score.
        return self.classes_[np.argmax(f_scores, axis=1)]


# --- 2. Subsampling Helper Functions (Ratio-Based) ---


def compute_centered_gram_matrix(X, gamma):
    """Computes a centered RBF Gram matrix.

    The kernel matrix is centered using the standard centering matrix:

        H = I - (1/n) * 11^T

    Args:
        X: Input feature matrix of shape (n_samples, n_features).
        gamma: RBF kernel gamma parameter.

    Returns:
        np.ndarray: Centered RBF Gram matrix of shape
            (n_samples, n_samples).
    """
    K = rbf_kernel(X, gamma=gamma)
    n = K.shape[0]

    # Construct the centering matrix H.
    H = np.eye(n) - (1.0 / n) * np.ones((n, n))

    # Apply double-centering to the kernel matrix.
    return H @ K @ H


def get_n_from_ratio(n_total, ratio, n_classes=10):
    """Calculates the number of selected samples from a dataset ratio.

    The returned sample count is the floor of `ratio * n_total`, with a
    minimum of one sample per class.

    Args:
        n_total: Total number of available training samples.
        ratio: Fraction of the training set to select.
        n_classes: Number of classes represented in the dataset.

    Returns:
        int: Number of samples to select.
    """
    return max(n_classes, int(np.floor(ratio * n_total)))


def get_all_samples_indices(y_train, V):
    """Returns all training sample indices for every view.

    Args:
        y_train: Training labels. Only the number of samples is used.
        V: Number of feature views.

    Returns:
        list[np.ndarray]: A list containing the complete index array
            for each view.
    """
    return [np.arange(len(y_train)) for _ in range(V)]


def get_global_random_indices(y_train, n_sub, V, rng):
    """Selects one shared random subset of samples for all views.

    The same randomly selected indices are copied to every view.

    Args:
        y_train: Training labels used to determine the dataset size.
        n_sub: Number of samples to select.
        V: Number of feature views.
        rng: NumPy random number generator.

    Returns:
        list[np.ndarray]: One identical random index array per view.
    """
    shared_indices = rng.choice(
        len(y_train),
        size=n_sub,
        replace=False,
    )

    return [shared_indices.copy() for _ in range(V)]


def get_uniform_random_indices(y_train, n_sub, V, rng):
    """Selects an independent random subset for each feature view.

    Args:
        y_train: Training labels used to determine the dataset size.
        n_sub: Number of samples to select per view.
        V: Number of feature views.
        rng: NumPy random number generator.

    Returns:
        list[np.ndarray]: Independently sampled indices for each view.
    """
    return [
        rng.choice(len(y_train), size=n_sub, replace=False)
        for _ in range(V)
    ]


def get_nystrom_indices(y_train, n_sub, V, rng):
    """Selects random landmark samples for Nyström approximation.

    This implementation uses uniform random sampling of training samples
    as the landmark selection strategy.

    Args:
        y_train: Training labels used to determine the dataset size.
        n_sub: Number of landmark samples to select.
        V: Number of feature views.
        rng: NumPy random number generator.

    Returns:
        list[np.ndarray]: Random landmark indices for each view.
    """
    return [
        rng.choice(len(y_train), size=n_sub, replace=False)
        for _ in range(V)
    ]


def get_kmeans_centroid_indices(views_train, n_sub):
    """Selects representative samples using K-Means cluster centroids.

    For each feature view, K-Means is fitted with `n_sub` clusters.
    The training sample closest to each cluster centroid is selected.

    Args:
        views_train: Dictionary mapping view names to training feature
            matrices.
        n_sub: Number of representative samples to select per view.

    Returns:
        list[np.ndarray]: Selected training indices for each view.
    """
    selected = []

    for _, X_v in views_train.items():
        # Cluster the samples into n_sub groups.
        kmeans = KMeans(
            n_clusters=n_sub,
            n_init=3,
            random_state=42,
        ).fit(X_v)

        closest_idx = []

        # Find the nearest unique training sample to each centroid.
        for c in kmeans.cluster_centers_:
            dists = np.linalg.norm(X_v - c, axis=1)

            for idx in np.argsort(dists):
                if idx not in closest_idx:
                    closest_idx.append(idx)
                    break

        selected.append(np.array(closest_idx, dtype=int))

    return selected


def get_stratified_kernel_herding_indices(
    views_train,
    y_train,
    gammas,
    n_sub,
):
    """Selects samples using class-stratified kernel herding.

    The target number of samples is distributed approximately uniformly
    across classes. Within each class and view, samples are selected
    greedily according to their kernel herding score.

    Args:
        views_train: Dictionary mapping view names to training feature
            matrices.
        y_train: Training class labels.
        gammas: Dictionary mapping view names to RBF kernel gamma values.
        n_sub: Total number of samples to select per view.

    Returns:
        list[np.ndarray]: Selected training indices for each view.
    """
    classes = np.unique(y_train)

    # Allocate approximately the same number of samples to each class.
    m_per_class = max(1, n_sub // len(classes))

    view_names = list(views_train.keys())
    selected_indices = {v_name: [] for v_name in view_names}

    for cls in classes:
        cls_mask = y_train == cls
        cls_indices = np.where(cls_mask)[0]
        n_cls = len(cls_indices)

        if n_cls == 0:
            continue

        for v_name in view_names:
            # Restrict the kernel herding computation to the current class.
            X_v_cls = views_train[v_name][cls_mask]

            K = rbf_kernel(
                X_v_cls,
                gamma=gammas[v_name],
            )

            # Mean kernel similarity used as the initial herding criterion.
            mu_k = np.mean(K, axis=1)
            chosen = []

            for t in range(min(m_per_class, n_cls)):
                if len(chosen) == 0:
                    # Select the sample with the largest mean similarity.
                    next_idx = np.argmax(mu_k)
                else:
                    # Greedy kernel herding score:
                    #
                    #   (t + 1) * mean_kernel - similarity_to_selected
                    #
                    # Already selected samples are excluded.
                    score = (
                        (t + 1) * mu_k
                        - np.sum(K[:, chosen], axis=1)
                    )
                    score[chosen] = -np.inf
                    next_idx = np.argmax(score)

                chosen.append(next_idx)

            # Convert class-local indices back to global training indices.
            selected_indices[v_name].extend(cls_indices[chosen])

    return [
        np.array(selected_indices[v_name], dtype=int)
        for v_name in view_names
    ]


def get_proposed_hsic_herding_indices(
    views_train,
    y_train,
    gammas,
    n_sub,
    warmup_ratio=0.1,
    random_state=42,
):
    """Selects samples using class-stratified HSIC-inspired kernel herding.

    A configurable fraction of samples is first selected randomly within
    each class as a warmup set. The remaining samples are selected using
    a greedy centered-kernel herding criterion.

    Args:
        views_train: Dictionary mapping view names to training feature
            matrices.
        y_train: Training class labels.
        gammas: Dictionary mapping view names to RBF kernel gamma values.
        n_sub: Total number of samples to select per view.
        warmup_ratio: Fraction of the per-class selection budget allocated
            to random warmup samples.
        random_state: Seed used for reproducible random sampling.

    Returns:
        list[np.ndarray]: Selected training indices for each view.
    """
    rng = np.random.default_rng(random_state)

    classes = np.unique(y_train)

    # Allocate the target number of samples approximately uniformly
    # across the available classes.
    m_per_class = max(1, n_sub // len(classes))

    # Determine how many samples per class are selected randomly
    # before applying the greedy herding procedure.
    w_per_class = int(m_per_class * warmup_ratio)

    view_names = list(views_train.keys())
    selected_indices = {v_name: [] for v_name in view_names}

    for cls in classes:
        cls_mask = y_train == cls
        cls_indices = np.where(cls_mask)[0]
        n_cls = len(cls_indices)

        if n_cls == 0:
            continue

        # Compute centered kernel matrices independently for each view.
        grams = [
            compute_centered_gram_matrix(
                views_train[v_name][cls_mask],
                gammas[v_name],
            )
            for v_name in view_names
        ]

        for v_idx, v_name in enumerate(view_names):
            K = grams[v_idx]
            chosen = []

            # Random warmup selection encourages an initial diverse subset.
            if w_per_class > 0:
                chosen.extend(
                    rng.choice(
                        n_cls,
                        size=min(w_per_class, n_cls),
                        replace=False,
                    ).tolist()
                )

            # Compute the mean centered-kernel similarity for each sample.
            mu_k = np.mean(K, axis=1)

            for t in range(
                len(chosen),
                min(m_per_class, n_cls),
            ):
                if len(chosen) == 0:
                    # If no warmup samples were selected, start with the
                    # sample having the largest mean kernel value.
                    next_idx = np.argmax(mu_k)
                else:
                    # Greedy kernel herding criterion after warmup.
                    score = (
                        (t + 1) * mu_k
                        - np.sum(K[:, chosen], axis=1)
                    )

                    # Prevent already selected samples from being chosen again.
                    score[chosen] = -np.inf
                    next_idx = np.argmax(score)

                chosen.append(next_idx)

            # Convert class-local indices to global training indices.
            selected_indices[v_name].extend(cls_indices[chosen])

    return [
        np.array(selected_indices[v_name], dtype=int)
        for v_name in view_names
    ]


# --- 3. Warmup Ratio Cross-Validation ---


def tune_hsic_warmup_ratio(
    views_train,
    y_train,
    gammas_list,
    n_sub,
    candidate_alphas=[0.0, 0.05, 0.1, 0.2, 0.3],
    cv=3,
):
    """Tunes the HSIC-herding warmup ratio using stratified cross-validation.

    Each candidate warmup ratio is evaluated using stratified K-fold
    cross-validation. For each fold, samples are selected using the
    proposed HSIC-herding strategy, an MV-LSSVM model is trained on the
    selected samples, and validation accuracy is measured.

    Args:
        views_train: Dictionary mapping view names to training feature
            matrices.
        y_train: Training class labels.
        gammas_list: List of RBF gamma values corresponding to the views.
        n_sub: Number of samples selected per view.
        candidate_alphas: Candidate warmup ratios to evaluate.
        cv: Number of stratified cross-validation folds.

    Returns:
        float: Candidate warmup ratio with the highest mean cross-validation
            accuracy.
    """
    # Preserve class proportions across all cross-validation folds.
    skf = StratifiedKFold(
        n_splits=cv,
        shuffle=True,
        random_state=42,
    )

    view_names = list(views_train.keys())

    # Initialize the best candidate using the first supplied ratio.
    best_alpha = candidate_alphas[0]
    best_score = -1.0

    for alpha in candidate_alphas:
        cv_scores = []

        # Generate stratified training and validation folds.
        for train_fold_idx, val_fold_idx in skf.split(
            views_train[view_names[0]],
            y_train,
        ):
            # Extract the current fold for every feature view.
            v_tr_fold = {
                v: views_train[v][train_fold_idx]
                for v in view_names
            }

            v_val_fold = [
                views_train[v][val_fold_idx]
                for v in view_names
            ]

            y_tr_fold = y_train[train_fold_idx]
            y_val_fold = y_train[val_fold_idx]

            # Select a subset of training samples using the candidate
            # warmup ratio.
            sub_idx = get_proposed_hsic_herding_indices(
                v_tr_fold,
                y_tr_fold,
                dict(zip(view_names, gammas_list)),
                n_sub,
                warmup_ratio=alpha,
                random_state=42,
            )

            # Train MV-LSSVM using the selected samples.
            clf = Multiclass_MV_LSSVM(
                kernel_gammas=gammas_list,
                selected_indices=sub_idx,
            )

            clf.fit(
                [v_tr_fold[v] for v in view_names],
                y_tr_fold,
            )

            # Evaluate the fitted model on the validation fold.
            preds = clf.predict(v_val_fold)
            cv_scores.append(
                accuracy_score(y_val_fold, preds)
            )

        # Compute the mean validation accuracy for this warmup ratio.
        mean_acc = np.mean(cv_scores)

        # Keep the ratio with the highest mean cross-validation accuracy.
        if mean_acc > best_score:
            best_score = mean_acc
            best_alpha = alpha

    return best_alpha
# --- 4. Dataset Loading & Setup ---

print("\n>>> Loading Digits dataset...")
digits = load_digits(n_class=10)
X_all, y_all = digits.data, digits.target

train_idx, test_idx = train_test_split(np.arange(len(y_all)), test_size=0.3, stratify=y_all, random_state=42)
X_images = X_all.reshape(-1, 8, 8)

views_tr = {f"Row_{r+1}": X_images[train_idx, r, :] for r in range(8)}
views_te = {f"Row_{r+1}": X_images[test_idx, r, :] for r in range(8)}
y_tr, y_te = y_all[train_idx], y_all[test_idx]

N_train = len(y_tr)
view_names = list(views_tr.keys())
V = len(view_names)
X_tr_list = [views_tr[v] for v in view_names]
X_te_list = [views_te[v] for v in view_names]

gammas_dict = {v_name: 1.0 / (views_tr[v_name].shape[1] * views_tr[v_name].var()) for v_name in view_names}
gammas_list = [gammas_dict[v] for v in view_names]


# ==============================================================================
# SECTION A: MV-LSSVM RATIO EXPERIMENT RUNNER
# ==============================================================================

# m represents the ratio of training dataset size per view (5%, 10%, 15%, 20%, 25%)
m_ratios = [0.05, 0.10, 0.15, 0.20, 0.25]
n_runs = 3

strategies = [
    "MV-LSSVM: All samples",
    "MV-LSSVM: Global Random (Same samples removed)",
    "MV-LSSVM: Uniform Random (Per view)",
    "MV-LSSVM: Nyström (Per view)",
    "MV-LSSVM: Unsupervised clustering",
    "MV-LSSVM: Stratified Kernel Herding",
    "MV-LSSVM: Proposed Stratified HSIC Herding"
]

results = {s: {m: [] for m in m_ratios} for s in strategies}
fit_times = {s: {m: [] for m in m_ratios} for s in strategies}
selected_alphas = {m: [] for m in m_ratios}

print(f"\n>>> Running MV-LSSVM Benchmark over Dataset Size Ratios m in {m_ratios}...")
print(f"    (Total training samples N_train = {N_train})")

for m in m_ratios:
    n_sub = get_n_from_ratio(N_train, m)
    
    # Cross-validation for warmup ratio alpha given the specific subsample budget size n_sub
    best_alpha = tune_hsic_warmup_ratio(views_tr, y_tr, gammas_list, n_sub, candidate_alphas=[0.0, 0.05, 0.1, 0.2, 0.3])
    selected_alphas[m] = best_alpha
    print(f"    [Ratio m={m:.2f} -> {n_sub} samples/view] Tuned Warmup Alpha: {best_alpha}")

    for run in range(n_runs):
        rng = np.random.default_rng(run)
        
        sub_samples = {
            "MV-LSSVM: All samples": get_all_samples_indices(y_tr, V),
            "MV-LSSVM: Global Random (Same samples removed)": get_global_random_indices(y_tr, n_sub, V, rng),
            "MV-LSSVM: Uniform Random (Per view)": get_uniform_random_indices(y_tr, n_sub, V, rng),
            "MV-LSSVM: Nyström (Per view)": get_nystrom_indices(y_tr, n_sub, V, rng),
            "MV-LSSVM: Unsupervised clustering": get_kmeans_centroid_indices(views_tr, n_sub),
            "MV-LSSVM: Stratified Kernel Herding": get_stratified_kernel_herding_indices(views_tr, y_tr, gammas_dict, n_sub),
            "MV-LSSVM: Proposed Stratified HSIC Herding": get_proposed_hsic_herding_indices(
                views_tr, y_tr, gammas_dict, n_sub, warmup_ratio=best_alpha, random_state=run
            )
        }

        for strat in strategies:
            t0 = time.time()
            clf = Multiclass_MV_LSSVM(kernel_gammas=gammas_list, selected_indices=sub_samples[strat])
            clf.fit(X_tr_list, y_tr)
            preds = clf.predict(X_te_list)
            elapsed = time.time() - t0

            results[strat][m].append(1.0 - accuracy_score(y_te, preds))
            fit_times[strat][m].append(elapsed)

# --- Save CSV Results ---
csv_rows = []
for strat in strategies:
    for m in m_ratios:
        n_sub = get_n_from_ratio(N_train, m)
        csv_rows.append({
            "strategy": strat,
            "dataset_ratio_m": m,
            "num_samples_per_view": n_sub,
            "tuned_alpha": selected_alphas[m] if "Proposed" in strat else np.nan,
            "mean_error_rate": np.mean(results[strat][m]),
            "std_error_rate": np.std(results[strat][m]),
            "mean_execution_time_sec": np.mean(fit_times[strat][m]),
            "std_execution_time_sec": np.std(fit_times[strat][m])
        })

df_results = pd.DataFrame(csv_rows)
df_results.to_csv(os.path.join(OUTPUT_DIR, "mvlssvm_ratio_results.csv"), index=False)


# ==============================================================================
# SECTION B: VISUALIZATIONS
# ==============================================================================

styles = {
    "MV-LSSVM: All samples": ("--", "black", 2.0),
    "MV-LSSVM: Global Random (Same samples removed)": (":", "gray", 1.5),
    "MV-LSSVM: Uniform Random (Per view)": ("-.", "orange", 1.5),
    "MV-LSSVM: Nyström (Per view)": (":", "blue", 1.5),
    "MV-LSSVM: Unsupervised clustering": ("--", "purple", 1.5),
    "MV-LSSVM: Stratified Kernel Herding": ("-.", "green", 1.8),
    "MV-LSSVM: Proposed Stratified HSIC Herding": ("-", "red", 2.5)
}

# --- Plot 1: Classification Error Rate vs Dataset Ratio m ---
plt.figure(figsize=(10, 5.5))
for strat in strategies:
    means = [np.mean(results[strat][m]) for m in m_ratios]
    ls, col, lw = styles[strat]
    plt.plot(m_ratios, means, linestyle=ls, color=col, linewidth=lw, marker="o", label=strat)

plt.title("MV-LSSVM Performance: Subsample Dataset Ratio ($m$) vs. Error Rate")
plt.xlabel("Subsample Ratio ($m = N_{\text{sub}} / N_{\text{train}}$)")
plt.ylabel("Test Classification Error Rate")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend(bbox_to_anchor=(1.04, 1), loc="upper left")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "mvlssvm_ratio_error.png"), dpi=300)
plt.close()

# --- Plot 2: Execution Time vs Dataset Ratio m ---
plt.figure(figsize=(10, 5.5))
for strat in strategies:
    t_means = [np.mean(fit_times[strat][m]) for m in m_ratios]
    ls, col, lw = styles[strat]
    plt.plot(m_ratios, t_means, linestyle=ls, color=col, linewidth=lw, marker="s", label=strat)

plt.title("MV-LSSVM Runtime: Subsample Dataset Ratio ($m$) vs. Execution Time")
plt.xlabel("Subsample Ratio ($m = N_{\text{sub}} / N_{\text{train}}$)")
plt.ylabel("Execution Time (Seconds)")
plt.yscale("log")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend(bbox_to_anchor=(1.04, 1), loc="upper left")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "mvlssvm_ratio_runtime.png"), dpi=300)
plt.close()


# ==============================================================================
# SECTION C: SUMMARY TABLE
# ==============================================================================
print("\n" + "=" * 92)
print(f"{'MV-LSSVM Strategy':<42} | {'m=0.05 Err':<10} | {'m=0.25 Time':<12} | {'m=0.25 Err':<10} | {'Alpha':<8}")
print("=" * 92)
for strat in strategies:
    e05 = np.mean(results[strat][0.05])
    e25 = np.mean(results[strat][0.25])
    t25 = np.mean(fit_times[strat][0.25])
    a_str = f"{selected_alphas[0.25]:.2f}" if "Proposed" in strat else "N/A"
    print(f"{strat:<42} | {e05:<10.4f} | {t25:<12.4f} | {e25:<10.4f} | {a_str:<8}")
print("=" * 92)

print(f"\n[SUCCESS] Ratio-based MV-LSSVM experiments complete. Saved to '{OUTPUT_DIR}'")

[INFO] Artifacts will be saved to: /Users/blancacanocamarero/tesis/repositories/subsampling-for-multi-view-kernel-methods/0_preliminar/MVLSVM/mvlssvm_ratio_results

>>> Loading Digits dataset...

>>> Running MV-LSSVM Benchmark over Dataset Size Ratios m in [0.05, 0.1, 0.15, 0.2, 0.25]...
    (Total training samples N_train = 1257)
    [Ratio m=0.05 -> 62 samples/view] Tuned Warmup Alpha: 0.2
    [Ratio m=0.10 -> 125 samples/view] Tuned Warmup Alpha: 0.1
    [Ratio m=0.15 -> 188 samples/view] Tuned Warmup Alpha: 0.1
    [Ratio m=0.20 -> 251 samples/view] Tuned Warmup Alpha: 0.1
    [Ratio m=0.25 -> 314 samples/view] Tuned Warmup Alpha: 0.0

MV-LSSVM Strategy                          | m=0.05 Err | m=0.25 Time  | m=0.25 Err | Alpha   
MV-LSSVM: All samples                      | 0.0796     | 4.2646       | 0.0796     | N/A     
MV-LSSVM: Global Random (Same samples removed) | 0.2735     | 0.1514       | 0.1099     | N/A     
MV-LSSVM: Uniform Random (Per view)        | 0.1605     | 0.161